# Rxrx3C phenotypic activity (copairs)

Per-perturbation cross-plate replicate-vs-control mAP using
[copairs](https://github.com/cytomining/copairs).

Rxrx3C has ~120 wells per perturbation across 9 plates — well-level copairs
saturates at sig=100% and takes ~7h to run. We aggregate to **per-
(perturbation × plate) consensus profiles** before running copairs (standard
JUMP-CP 'consensus profile' mAP setup). This gives ~9 cross-plate replicates
per perturbation and one control consensus per plate.

Pairing rules (cross-plate replicate consistency):
- **Positive**: same `perturbation`, *different* plate (`pos_diffby=['batch','plate']`).
- **Negative**: same physical plate (`batch` × `plate`), one is the control consensus (EMPTY).

Headline metric: fraction of perturbations whose mAP is significant after
BH multiple-testing correction (`below_corrected_p.mean()`). Reported on both
pre-Harmony (`X_pca`) and post-Harmony (`X_pca_harmony`) embeddings.

Held-out batch protocol from the recall eval is **not** applied here.

In [ ]:
from __future__ import annotations

import time
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from copairs import map as cmap
from copairs.matching import assign_reference_index

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [ ]:
EMBED_DIR = Path('../data')
EMBEDDING_KEYS = ('X_pca', 'X_pca_harmony')

DATASET = 'Rxrx3C'
PERTURBATION_COL = 'perturbation'
CONTROL_COL = 'is_control'
PLATE_COLS = ('batch', 'plate')  # batch=plate_N (= split unit), plate=1..9

NULL_SIZE = 10000
P_THRESHOLD = 0.05
BATCH_SIZE = 20000

GROUPS: dict[str, dict[str, str]] = {
    'Rxrx3C — DINO': {
        'C':  '25_Rxrx3C_DINO_ESM2_C_aggregated.h5ad',
        'CD': '26_Rxrx3C_DINO_ESM2_CD_aggregated.h5ad',
        'S':  '27_Rxrx3C_DINO_ESM2_S_aggregated.h5ad',
        'SD': '28_Rxrx3C_DINO_ESM2_SD_aggregated.h5ad',
    },
    'Rxrx3C — OpenPhenom': {
        'C':  '29_Rxrx3C_OpenPhenom_ESM2_C_aggregated.h5ad',
        'CD': '30_Rxrx3C_OpenPhenom_ESM2_CD_aggregated.h5ad',
        'S':  '31_Rxrx3C_OpenPhenom_ESM2_S_aggregated.h5ad',
        'SD': '32_Rxrx3C_OpenPhenom_ESM2_SD_aggregated.h5ad',
        'zero-shot': 'OpenPhenom_zeroshot_Rxrx3C_aggregated.h5ad',
    },
    'Rxrx3C — SubCell': {
        'C':  '33_Rxrx3C_SubCell_ESM2_C_aggregated.h5ad',
        'CD': '34_Rxrx3C_SubCell_ESM2_CD_aggregated.h5ad',
        'S':  '35_Rxrx3C_SubCell_ESM2_S_aggregated.h5ad',
        'SD': '36_Rxrx3C_SubCell_ESM2_SD_aggregated.h5ad',
    },
}

In [ ]:
def consensus_profiles(
    adata: ad.AnnData,
    embedding_key: str,
) -> tuple[pd.DataFrame, np.ndarray]:
    """Aggregate to per-(batch, plate, perturbation, is_control) mean profile.

    Returns the metadata frame (with `Metadata_*` columns) and a (n, d) feature
    matrix in the same row order. Each treated perturbation contributes up to
    one profile per plate; controls contribute one profile per plate.
    """
    feats = np.asarray(adata.obsm[embedding_key])
    obs = adata.obs[list(PLATE_COLS) + [PERTURBATION_COL, CONTROL_COL]].astype(str).reset_index(drop=True)
    feat_cols = [f'feat_{i}' for i in range(feats.shape[1])]
    df = pd.concat([obs, pd.DataFrame(feats, columns=feat_cols)], axis=1)
    group_cols = list(PLATE_COLS) + [PERTURBATION_COL, CONTROL_COL]
    agg = df.groupby(group_cols, observed=True, as_index=False)[feat_cols].mean()
    meta = agg[group_cols].copy()
    meta.columns = [f'Metadata_{c}' for c in group_cols]
    fv = agg[feat_cols].to_numpy()
    return meta, fv

In [ ]:
def phenotypic_activity(
    adata: ad.AnnData,
    embedding_key: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Run copairs on per-(plate, perturbation) consensus profiles.

    Positive = same `perturbation`, *different* plate. Negative = same
    physical plate (`batch` x `plate`), one row is the control consensus.
    Controls excluded from the final mAP rollup.
    """
    meta, fv = consensus_profiles(adata, embedding_key)

    ref_col = 'Metadata_reference_index'
    meta = assign_reference_index(
        meta,
        f"Metadata_{CONTROL_COL} != 'True'",
        reference_col=ref_col,
        default_value=-1,
    )

    pos_sameby = [f'Metadata_{PERTURBATION_COL}']
    pos_diffby = [f'Metadata_{c}' for c in PLATE_COLS]  # cross-plate replicates
    neg_sameby = [f'Metadata_{c}' for c in PLATE_COLS]
    neg_diffby = [ref_col]

    ap = cmap.average_precision(
        meta=meta, feats=fv,
        pos_sameby=pos_sameby, pos_diffby=pos_diffby,
        neg_sameby=neg_sameby, neg_diffby=neg_diffby,
        batch_size=BATCH_SIZE,
    )
    ap = ap.query(f"Metadata_{CONTROL_COL} != 'True'")
    maps = cmap.mean_average_precision(
        ap, pos_sameby, null_size=NULL_SIZE, threshold=P_THRESHOLD, seed=RANDOM_SEED,
    )
    return maps, ap

In [ ]:
results: dict[str, dict[str, dict[str, pd.DataFrame]]] = {}
summary_rows = []

for group_name, view_files in GROUPS.items():
    print(f'=== {group_name} ===')
    results[group_name] = {}
    for view, fname in view_files.items():
        path = EMBED_DIR / fname
        adata = ad.read_h5ad(path)
        results[group_name][view] = {}
        for emb in EMBEDDING_KEYS:
            t0 = time.time()
            maps, _ = phenotypic_activity(adata, emb)
            results[group_name][view][emb] = maps
            tag = 'pre_harmony' if emb == 'X_pca' else 'post_harmony'
            sig_q = float(maps['below_corrected_p'].mean())
            sig_p = float(maps['below_p'].mean())
            mean_map = float(maps['mean_average_precision'].mean())
            summary_rows.append({
                'group': group_name,
                'view': view,
                'embedding': tag,
                'n_perturbations': len(maps),
                'mean_mAP': mean_map,
                'sig_frac_p': sig_p,
                'sig_frac_q': sig_q,
            })
            print(f'  {view:>9s} {tag:>13s}  n={len(maps):>5d}  mAP={mean_map:.3f}  '
                  f'sig(p)={sig_p:.3f}  sig(q)={sig_q:.3f}  ({time.time()-t0:.1f}s)')

summary = pd.DataFrame(summary_rows)
summary

In [ ]:
summary.to_csv(f'phenotypic_activity_{DATASET.lower()}.csv', index=False)
summary.pivot_table(
    index=['group', 'view'], columns='embedding',
    values=['mean_mAP', 'sig_frac_q'],
)

In [ ]:
VIEW_STYLES = {
    'C':         {'color': 'C0',   'hatch': None,    'alpha': 1.00},
    'CD':        {'color': 'C1',   'hatch': None,    'alpha': 1.00},
    'S':         {'color': 'C2',   'hatch': None,    'alpha': 1.00},
    'SD':        {'color': 'C3',   'hatch': None,    'alpha': 1.00},
    'zero-shot': {'color': 'gray', 'hatch': '//',    'alpha': 0.85},
}
VIEW_ORDER = ['C', 'CD', 'S', 'SD', 'zero-shot']

def plot_one(metric_col: str, ylabel: str, ymax: float | None, fname: str) -> None:
    n_groups = len(GROUPS)
    for emb in EMBEDDING_KEYS:
        tag = 'pre_harmony' if emb == 'X_pca' else 'post_harmony'
        fig, axes = plt.subplots(1, n_groups, figsize=(4.5 * n_groups, 4),
                                 sharey=True, squeeze=False)
        for ax, (group_name, view_files) in zip(axes[0], GROUPS.items()):
            views = [v for v in VIEW_ORDER if v in view_files]
            xs = np.arange(len(views))
            row_lookup = {(r['group'], r['view'], r['embedding']): r for r in summary_rows}
            ys = [row_lookup[(group_name, v, tag)][metric_col] for v in views]
            for x, v, y in zip(xs, views, ys):
                style = VIEW_STYLES[v]
                ax.bar(x, y, color=style['color'], hatch=style['hatch'],
                       alpha=style['alpha'], edgecolor='black', linewidth=0.7,
                       label=v)
                ax.text(x, y + 0.01, f'{y:.2f}', ha='center', va='bottom', fontsize=8)
            ax.set_xticks(xs)
            ax.set_xticklabels(views, rotation=0)
            ax.set_title(group_name.replace(f'{DATASET} — ', ''))
            ax.set_xlabel('view')
            ax.grid(axis='y', alpha=0.3)
        axes[0, 0].set_ylabel(ylabel)
        if ymax is not None:
            axes[0, 0].set_ylim(0, ymax)
        fig.suptitle(f'{DATASET} {ylabel} ({tag})', y=1.02)
        fig.tight_layout()
        out = f'{fname}_{DATASET.lower()}_{tag}.png'
        fig.savefig(out, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'  saved {out}')

plot_one('sig_frac_q', 'significant fraction (BH q<0.05)', 1.05, 'phenotypic_activity_sigfrac')
plot_one('mean_mAP', 'mean mAP', None, 'phenotypic_activity_mAP')